<a href="https://colab.research.google.com/github/rajeshwarisubramani/AIPractice/blob/main/Week_19_Day_1_2_Simple_Financial_Budgeting_Agent.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install langchain-openai --q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 122.2/122.2 kB 9.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 561.7/561.7 kB 41.5 MB/s eta 0:00:00


In [2]:
import warnings
warnings.filterwarnings("ignore")

In [3]:
import json
from typing import List, TypedDict
from pydantic import BaseModel, Field
from langgraph.graph import StateGraph, END

from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate


In [4]:
import os
os.environ['OPENAI_API_BASE'] = ""
os.environ['OPENAI_API_KEY'] = ""

In [5]:
# Initialize model
llm = ChatOpenAI(model="gpt-3.5-turbo", temperature=0)

In [6]:
class FinancialState(TypedDict):
    department: str
    historical_spend: dict  # Category -> Amount
    target_growth_rate: float
    projected_budget: dict
    variance_analysis: str
    final_report: str

In [7]:
def ingestion_agent(state: FinancialState) -> dict:
    """Agent 1: Ingests and standardizes historical financial data."""
    dept = state["department"]

    # Simulated data ingestion (e.g., from DB or ERP system)
    mock_db = {
        "Engineering": {"salaries": 500000, "cloud_infrastructure": 120000, "software_licenses": 30000},
        "Marketing": {"ad_campaigns": 200000, "salaries": 250000, "events": 50000}
    }

    data = mock_db.get(dept, {"salaries": 100000, "operational": 20000})
    return {"historical_spend": data}

In [8]:
def analyst_agent(state: FinancialState) -> dict:
    # 1. Deterministic Python Math (Never let the LLM do raw math)
    historical = state["historical_spend"]
    growth_rate = state["target_growth_rate"]
    projected = {cat: round(amt * (1 + growth_rate), 2) for cat, amt in historical.items()}

    # 2. LLM Reasoning for Qualitative Insights & Risk Flags
    prompt = ChatPromptTemplate.from_template(
        "Analyze this department budget for {department}.\n"
        "Historical: {historical}\n"
        "Projected: {projected}\n"
        "Write 2 concise executive bullet points summarizing budget risks "
        "and strategic implications of this {growth}% change."
    )

    chain = prompt | llm
    llm_response = chain.invoke({
        "department": state["department"],
        "historical": historical,
        "projected": projected,
        "growth": growth_rate * 100
    })

    return {
        "projected_budget": projected,
        "variance_analysis": llm_response.content
    }

In [9]:
def approver_agent(state: FinancialState) -> dict:
    """Agent 3: Compiles, validates, and formats the final budget report."""
    dept = state["department"]
    hist = state["historical_spend"]
    proj = state["projected_budget"]
    analysis = state["variance_analysis"]

    report_lines = [
        f"==========================================",
        f"       BUDGET REPORT: {dept.upper()}",
        f"==========================================",
        f"\n[Category Breakdown]",
        f"{'Category':<22} | {'Previous':<10} | {'Projected':<10}",
        f"------------------------------------------"
    ]

    for cat in hist:
        report_lines.append(f"{cat:<22} | ${hist[cat]:<9,.2f} | ${proj[cat]:<9,.2f}")

    report_lines.extend([
        f"------------------------------------------",
        f"TOTAL                  | ${sum(hist.values()):<9,.2f} | ${sum(proj.values()):<9,.2f}",
        f"\n[Financial Analysis]",
        analysis,
        f"\nStatus: APPROVED BY AUTOMATED PROCESSOR"
    ])

    return {"final_report": "\n".join(report_lines)}

In [10]:
workflow = StateGraph(FinancialState)

# Add agent nodes
workflow.add_node("IngestionAgent", ingestion_agent)
workflow.add_node("AnalystAgent", analyst_agent)
workflow.add_node("ApproverAgent", approver_agent)

# Define linear agent flow
workflow.set_entry_point("IngestionAgent")
workflow.add_edge("IngestionAgent", "AnalystAgent")
workflow.add_edge("AnalystAgent", "ApproverAgent")
workflow.add_edge("ApproverAgent", END)

In [11]:
budget_app = workflow.compile()

In [12]:
initial_input = {"department": "Engineering",
                 "target_growth_rate": 0.08  # 8% growth target
                }

In [13]:
result = budget_app.invoke(initial_input)

In [14]:
print(result["final_report"])

       BUDGET REPORT: ENGINEERING

[Category Breakdown]
Category               | Previous   | Projected 
------------------------------------------
salaries               | $500,000.00 | $540,000.00
cloud_infrastructure   | $120,000.00 | $129,600.00
software_licenses      | $30,000.00 | $32,400.00
------------------------------------------
TOTAL                  | $650,000.00 | $702,000.00

[Financial Analysis]
- Budget risks: The projected increase in salaries, cloud infrastructure, and software licenses indicates a potential strain on the department's financial resources. If not managed effectively, this could lead to budget overruns and potential cutbacks in other areas.

- Strategic implications: The 8.0% change in the budget highlights the need for the department to reassess its spending priorities and potentially explore cost-saving measures. This could involve renegotiating contracts with software vendors, optimizing cloud infrastructure usage, and evaluating staffing needs to e